# 06 — Structured Streaming in Apache Spark

## Streaming vs Batch — The Mental Model

In **batch processing**, a DataFrame has a fixed, known set of rows. You run a job, it finishes,
you get results.

In **streaming**, the DataFrame is **unbounded** — rows keep arriving over time. The query never
truly "finishes"; it runs continuously, processing new data as it appears.

Spark Structured Streaming bridges the gap with a **micro-batch model**:

```
Continuous stream  →  [Trigger interval]  →  micro-batch  →  run as a regular Spark job  →  write results
                            ↓ (repeat)
```

Key insight: **you write the same DataFrame/SQL transformations you already know**. Spark handles
the bookkeeping of "what has already been processed" via a **checkpoint** directory.

**Cluster:** `spark://spark-master:7077`  
**Data dir:** `/home/jovyan/data`

## Structured Streaming Architecture

```
┌──────────────────────────────────────────────────────────────────┐
│                     Structured Streaming Engine                  │
│                                                                  │
│  Input Source          Trigger           Output Sink             │
│  ─────────────         ───────           ───────────             │
│  • Kafka               • ProcessingTime  • Kafka                 │
│  • Files (CSV/JSON)    • Once            • Files (Parquet/CSV)   │
│  • Rate (testing)      • AvailableNow    • Console (debug)       │
│  • Socket (testing)    • Continuous      • Memory (testing)      │
│                                          • foreach / foreachBatch│
└──────────────────────────────────────────────────────────────────┘
```

### Output Modes

| Mode | Description | Requires | Typical Use |
|---|---|---|---|
| **append** | Only new rows since last trigger are written | No aggregation, or aggregation with watermark | Append-only event logs |
| **complete** | The entire result table is rewritten each trigger | Aggregation | Running totals, leaderboards |
| **update** | Only rows that changed since last trigger | Aggregation | Updating dashboard metrics |

### Exactly-Once Semantics

Spark guarantees **exactly-once** end-to-end processing when:
1. The **source** is replayable (Kafka, files — not sockets).
2. The **sink** is idempotent or transactional.
3. A **checkpoint** location is configured.

The checkpoint stores the offset range processed in each micro-batch. On restart, Spark reads
the checkpoint to know exactly where to resume — no row is processed twice.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.streaming import StreamingQuery
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
import os, time, pathlib

# -----------------------------------------------------------------------
# SparkSession with streaming-relevant configuration
#
# spark.sql.streaming.schemaInference   — allow schema inference on file streams
# spark.sql.shuffle.partitions          — keep micro-batch shuffles small
# -----------------------------------------------------------------------
spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("06-Structured-Streaming-Intro")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.streaming.schemaInference", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Convenience paths
DATA_DIR        = pathlib.Path("/home/jovyan/data")
CHECKPOINT_DIR  = DATA_DIR / "checkpoints"
STREAM_CSV_DIR  = DATA_DIR / "stream_csv_input"

# Ensure directories exist
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
STREAM_CSV_DIR.mkdir(parents=True, exist_ok=True)

print(f"Spark version        : {spark.version}")
print(f"Data directory       : {DATA_DIR}")
print(f"Checkpoint directory : {CHECKPOINT_DIR}")

In [ ]:
# -----------------------------------------------------------------------
# Rate source — built-in synthetic stream, no external dependencies needed.
#
# The 'rate' source generates rows at a configurable rate:
#   column 'timestamp' : the event time of the row
#   column 'value'     : a monotonically increasing Long (0, 1, 2, ...)
#
# This is the easiest way to test streaming logic without Kafka.
# -----------------------------------------------------------------------

rate_stream = (
    spark.readStream
    .format("rate")                       # built-in synthetic generator
    .option("rowsPerSecond", 10)          # 10 rows/sec across all partitions
    .option("numPartitions", 4)           # 4 parallel generating tasks
    .load()
)

# Verify it is indeed a streaming DataFrame
print(f"isStreaming : {rate_stream.isStreaming}")
print(f"Schema      : {rate_stream.schema.simpleString()}")

# printSchema() works the same on streaming DataFrames
rate_stream.printSchema()

In [ ]:
# -----------------------------------------------------------------------
# Apply transformations — identical syntax to batch DataFrames.
#
# Here we:
#   1. Add a 'category' column by bucketing the value into 5 groups.
#   2. Add a 'processing_time' column stamped at processing time.
#   3. Add a 'lag_ms' column showing simulated processing latency.
#
# These are stateless transformations (no groupBy across batches).
# -----------------------------------------------------------------------

transformed_stream = (
    rate_stream
    .withColumn(
        "category",
        F.element_at(
            F.array(F.lit("alpha"), F.lit("beta"), F.lit("gamma"),
                    F.lit("delta"), F.lit("epsilon")),
            (F.col("value") % 5).cast("int") + 1
        )
    )
    .withColumn("processing_time", F.current_timestamp())
    .withColumn(
        "lag_ms",
        # Difference between processing time and event time in milliseconds
        (
            F.col("processing_time").cast("double")
            - F.col("timestamp").cast("double")
        ) * 1000
    )
)

print("Transformed stream schema:")
transformed_stream.printSchema()

In [ ]:
# -----------------------------------------------------------------------
# Write the stream to the 'memory' sink.
#
# The memory sink stores all results in a temporary in-memory table.
# It is ONLY intended for development and testing — not production.
#
# queryName  → the name of the in-memory table you can query with spark.sql()
# outputMode → 'append' means each micro-batch only adds new rows
# checkpointLocation → where Spark stores offset progress
# -----------------------------------------------------------------------

QUERY_NAME = "rate_stream_output"

query: StreamingQuery = (
    transformed_stream.writeStream
    .format("memory")                                         # in-memory sink
    .queryName(QUERY_NAME)                                    # table name
    .outputMode("append")                                     # only new rows each batch
    .option("checkpointLocation",
            str(CHECKPOINT_DIR / "rate_stream"))              # fault-tolerance
    .start()
)

print(f"Query name    : {query.name}")
print(f"Query ID      : {query.id}")
print(f"Query status  : {query.status}")

# Let the stream run for 5 seconds to accumulate some rows
print("\nCollecting data for 5 seconds...")
query.awaitTermination(timeout=5)   # returns after timeout; stream keeps running

# Query the in-memory table just like a regular table
result_df = spark.sql(f"SELECT * FROM {QUERY_NAME} ORDER BY timestamp")
print(f"\nRows collected so far: {result_df.count()}")
result_df.show(10, truncate=False)

# Let it run a few more seconds then peek again
time.sleep(3)
print(f"Rows after 3 more seconds: {spark.sql(f'SELECT COUNT(*) as n FROM {QUERY_NAME}').collect()[0].n}")

# Stop this specific query
query.stop()
print("\nStreaming query stopped.")

In [ ]:
# -----------------------------------------------------------------------
# Trigger types — control when each micro-batch runs
#
# Trigger.ProcessingTime(interval)  — run a batch every N seconds/minutes
# Trigger.Once()                    — run exactly ONE batch then stop (deprecated in 3.3)
# Trigger.AvailableNow()            — process all available data as fast as possible,
#                                     then stop — the modern replacement for Trigger.Once()
# Trigger.Continuous(interval)      — low-latency (~1ms) continuous processing;
#                                     experimental, limited operators
# -----------------------------------------------------------------------

from pyspark.sql.streaming import Trigger   # available since Spark 3.x

# We demonstrate each Trigger by just printing what we would write —
# actually starting all of them would flood the cluster.

trigger_examples = {
    "ProcessingTime('10 seconds')": (
        "Runs a micro-batch every 10 seconds. "
        "If the batch takes longer than 10s, the next one starts immediately."
    ),
    "Once()": (
        "Deprecated since Spark 3.3. Runs exactly one micro-batch then stops. "
        "Useful for scheduled (non-continuous) stream processing."
    ),
    "AvailableNow()": (
        "Replacement for Trigger.Once(). Processes ALL available data in "
        "multiple micro-batches (for better scalability), then stops automatically."
    ),
    "Continuous('1 second')": (
        "Experimental low-latency mode. Millisecond-level latency. "
        "Supports only simple map/filter operations — no aggregations."
    ),
}

print("Trigger type comparison:\n")
for name, description in trigger_examples.items():
    print(f"  Trigger.{name}")
    print(f"    → {description}")
    print()

# --- Concrete example: Trigger.AvailableNow() ----------------------------
# This is the recommended way to run "batch-like" incremental processing.

rate_for_trigger = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 50)
    .load()
)

q_available_now = (
    rate_for_trigger.writeStream
    .format("memory")
    .queryName("trigger_demo")
    .outputMode("append")
    .trigger(availableNow=True)    # process all backlog then stop
    .option("checkpointLocation", str(CHECKPOINT_DIR / "trigger_demo"))
    .start()
)

# awaitTermination() with no timeout blocks until the query finishes naturally
q_available_now.awaitTermination(timeout=15)
print(f"Trigger.AvailableNow() finished. Rows: "
      f"{spark.sql('SELECT COUNT(*) AS n FROM trigger_demo').collect()[0].n}")

In [ ]:
# -----------------------------------------------------------------------
# File source stream — read CSV files as a continuous stream.
#
# Spark monitors a directory and processes each new file as a micro-batch.
# Files already present when the query starts are treated as historical data.
# New files dropped into the directory are picked up at the next trigger.
#
# Requirements:
#   1. A schema must be provided explicitly (inference is unreliable for streams).
#   2. Files must NOT be modified after they are written (append-only source).
# -----------------------------------------------------------------------

# ---- Step 1: Write some initial CSV files to simulate an input feed ----

csv_schema = StructType([
    StructField("event_id",   StringType(),  nullable=False),
    StructField("event_type", StringType(),  nullable=True),
    StructField("amount",     DoubleType(),  nullable=True),
])

import csv, uuid, random

event_types = ["purchase", "refund", "view", "click", "signup"]

for file_num in range(3):   # write 3 initial CSV files
    file_path = STREAM_CSV_DIR / f"events_batch_{file_num:03d}.csv"
    with open(file_path, "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["event_id", "event_type", "amount"])  # header
        for _ in range(100):   # 100 rows per file
            writer.writerow([
                str(uuid.uuid4()),
                random.choice(event_types),
                round(random.uniform(1.0, 500.0), 2)
            ])
    print(f"Written: {file_path}")

# ---- Step 2: Set up the streaming read -------------------------------------

csv_stream = (
    spark.readStream
    .format("csv")
    .schema(csv_schema)                        # schema must be explicit
    .option("path", str(STREAM_CSV_DIR))       # directory to watch
    .option("header", "true")                  # first row is a header
    .option("maxFilesPerTrigger", 1)           # process one file per micro-batch
    .load()
)

print(f"\nCSV stream isStreaming: {csv_stream.isStreaming}")
csv_stream.printSchema()

# ---- Step 3: Aggregate — total amount per event_type ----------------------

csv_agg = (
    csv_stream
    .groupBy("event_type")
    .agg(
        F.count("*").alias("event_count"),
        F.round(F.sum("amount"), 2).alias("total_amount")
    )
)

# ---- Step 4: Write to memory sink with 'complete' output mode -------------
# complete is required for aggregations without a watermark

q_csv = (
    csv_agg.writeStream
    .format("memory")
    .queryName("csv_stream_agg")
    .outputMode("complete")   # rewrite entire result table each trigger
    .option("checkpointLocation", str(CHECKPOINT_DIR / "csv_stream"))
    .trigger(availableNow=True)
    .start()
)

q_csv.awaitTermination(timeout=30)

print("\nAggregated results from CSV stream:")
spark.sql("""
    SELECT event_type, event_count, total_amount
    FROM csv_stream_agg
    ORDER BY total_amount DESC
""").show(truncate=False)

In [ ]:
# -----------------------------------------------------------------------
# Stop all active streaming queries and the SparkSession.
#
# Best practice: always stop queries explicitly to release executor
# threads and network connections back to the cluster.
# -----------------------------------------------------------------------

# Stop any remaining active queries
for active_query in spark.streams.active:
    print(f"Stopping query: {active_query.name} ({active_query.id})")
    active_query.stop()

print(f"Active queries after stop: {len(spark.streams.active)}")

spark.stop()
print("SparkSession stopped. Resources released.")